In [ ]:
# 1. Install necessary libraries
!pip install -q -U google-generativeai pandas

# 2. Imports and Setup
import os
import json
import pandas as pd
import time
import google.generativeai as genai
from google.colab import drive
from IPython.display import display, Markdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 35.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# --- CONFIGURATION ---
# Mount Google Drive
drive.mount('/content/drive')

# API Key Setup (Get this from aistudio.google.com)
# Best practice: Use Colab Secrets (the key icon on the left sidebar)
from google.colab import userdata
# Replace 'YOUR_SECRET_NAME_HERE' with the actual name you used for your API key in Colab Secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') # Example: userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

Mounted at /content/drive


In [ ]:
# Define the folder containing your PDFs
EXAM_FOLDER_PATH = '/content/drive/MyDrive/Grading/GEO315_Midterm_Exam'

In [ ]:
# The Answer Key (Pasted from your prompt)
ANSWER_KEY = """
Part 1: True/False and Multiple Choice
1.	True
2.	False
o	Note: Strain is dimensionless (change in length over original length); Stress is defined as force over area.
3.	True
4.	False
o	Note: The person wearing heels exerts force over a much smaller area, resulting in greater stress.
5.	False
o	Note: For any normal fault, the principal compressive stress axis ($\sigma_1$) is oriented vertically, not horizontally.
6.	True
7.	True
o	Note: The joints shown are parallel and evenly spaced, classifying them as systematic.
8.	True
9.	True
10.	False
o	Note: Detachment faults are characterized by a low dip angle.
11.	The description and analysis of the movement and deformation of Earth's crust without regard to the forces involved.
12.	The study of the forces and stresses that lead to geological structures.
13.	The force applied per unit area within a geological structure.
14.	All of the above
15.	Rubber
16.	Sandstone
o	Note: The sandstone layer lacks the closely spaced systematic fractures seen in the dolomite, indicating it was able to accommodate the strain elastically (lower Young's Modulus).
17.	Top-to-the-right
18.	Into a roof thrust
19.	Cataclasite
20.	Ultramylonite
________________________________________
Part 2: Short Answer
21. Geologic History
•	Youngest (1): Dike Intrusion (the dark band cuts continuously through all other features)
•	2: Faulting (offsets the folded layers but does not offset the dike)
•	3: Folding (bends the sedimentary layers prior to faulting)
•	Oldest (4): Deposition of sedimentary layers
22. Block Diagram Sketch Instructions
•	(a) Draw parallel lines running straight east-west on the top surface. On the front face, draw lines dipping uniformly to the right to represent the 30° dip.
•	(b) Draw lines striking northwest-southeast on the top surface. On the front and side faces, draw vertical straight lines going straight down to represent the 90° dip.
•	(c) Draw parallel lines running straight east-west on the top surface. On the front face, draw lines dipping uniformly to the left to represent the 30° dip into the page/block.
•	(d) Draw lines striking northeast-southwest on the top surface. On the front face, draw shallowly dipping lines (10°) angling to the right.
23. Stress and Faulting Explanation According to Anderson's theory of faulting, the normal stress ($\sigma_n$) acting on a fault surface strongly controls its frictional strength. For a reverse fault, the maximum principal stress ($\sigma_1$) is horizontal, and the minimum principal stress ($\sigma_3$) is vertical (determined by the lithostatic pressure of overlying rock). Conversely, for a normal fault, the vertical lithostatic pressure acts as $\sigma_1$. Because the horizontal $\sigma_1$ required to initiate a reverse fault must overcome both friction and the weight of the rock ($\sigma_3$), the normal stress clamped across a reverse fault is generally much higher than on a normal fault at the same depth. Consequently, a higher shear stress is required to overcome the frictional resistance and initiate sliding.
24. Mohr Diagram Problem
•	Approximate dip angle: $45^\circ$
•	Derivation: We know that the fault locks when the ratio of shear stress to normal stress falls below the coefficient of friction:
$$\sigma_s / \sigma_n = \mu$$
Given $\mu = 0.5$, the failure envelope is defined by $\sigma_s = 0.5\sigma_n$. The center of the Mohr circle is at $C = (\sigma_1 + \sigma_3) / 2 = 40$ MPa, and the radius is $R = (\sigma_1 - \sigma_3) / 2 = 20$ MPa. Using the Mohr circle equations for stress on a plane:
$$\sigma_n = 40 + 20\cos(2\theta)$$
$$\sigma_s = 20\sin(2\theta)$$
Setting $\sigma_s = 0.5\sigma_n$ and substituting the circle constraints reveals that the circle intersects the failure envelope at $\sigma_n = 40$ MPa and $\sigma_s = 20$ MPa.
Substituting this back into the normal stress equation:
$$40 = 40 + 20\cos(2\theta)$$
$$\cos(2\theta) = 0$$
$$2\theta = 90^\circ$$
$$\theta = 45^\circ$$
As the fault rotates during bookshelf faulting, its dip becomes shallower. Once the dip reaches $45^\circ$, the stress point falls below the friction envelope, and the fault locks up.
25. Thrust Wedge and Duplex Sketches
•	Imbricate Fan (Thrust Wedge): Ensure the sketch shows a series of listric (curving) reverse faults that branch upward off a single basal detachment (floor thrust) and breach the surface. Add half-arrows showing top-to-the-foreland (hanging wall moving up) displacement.
•	Thrust Duplex: Ensure the sketch depicts a floor thrust at the base and a roof thrust at the top. Between these parallel boundaries, sketch multiple sigmoidal imbricate faults (horses) that link the floor to the roof. Add reverse kinematic half-arrows on the horses, roof, and floor thrusts.
"""

<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_16382/4026139976.py:11: SyntaxWarning: invalid escape sequence '\s'
  o	Note: For any normal fault, the principal compressive stress axis ($\sigma_1$) is oriented vertically, not horizontally.


In [ ]:
# --- THE GRADING FUNCTION ---
def grade_exam(file_path):
    # Upload file to Gemini
    sample_file = genai.upload_file(path=file_path, display_name="Student Exam")

    # Set up the model
    model = genai.GenerativeModel('gemini-2.5-pro')

    # The Prompt - Enforcing JSON output
    prompt = f"""
    You are a Professor TA. Grade this attached student exam based on the following Answer Key. Where a graphical answer is sufficient to answer quantitative questions, accept the graphic answer. Award partial credit where deserved.

    Answer Key:
    {ANSWER_KEY}

    Instructions:
    1. Extract the Student Name from the exam.
    2. Grade Section 1 and Section 2 (Parts 1 and 2) based on the key.
    3. Calculate the Total Score.
    4. Create a comments field that logs if you were unable to confidently score any questions.
    5. Create a scoring breakdown including each question that points were deducted for, with a brief justification.

    IMPORTANT: specific output format.
    Return ONLY a raw JSON object. Do not use Markdown code blocks.
    The JSON must have these exact keys:
    {{
        "student_name": "Name",
        "section_1_score": 0,
        "section_2_score": 0,
        "total_score": 0,
        "comments": "Text",
        "scoring_breakdown": "Text"
    }}
    """

    try:
        response = model.generate_content([prompt, sample_file])
        # Clean response string to ensure pure JSON
        cleaned_response = response.text.replace('```json', '').replace('```', '').strip()
        return json.loads(cleaned_response)
    except Exception as e:
        print(f"Error grading {file_path}: {e}")
        return None

In [ ]:
output_csv = os.path.join(EXAM_FOLDER_PATH, 'Graded_Results.csv')
processed_files = set()

# Load existing results if the file exists
if os.path.exists(output_csv):
    try:
        existing_df = pd.read_csv(output_csv)
        # Assuming your CSV has a 'filename' column
        if 'filename' in existing_df.columns:
            processed_files = set(existing_df['filename'].unique())
        print(f"Loaded {len(processed_files)} previously graded exams.")
    except Exception as e:
        print(f"Could not read existing CSV: {e}")

# --- PROCESSING LOOP ---
if os.path.exists(EXAM_FOLDER_PATH):
    files = [f for f in os.listdir(EXAM_FOLDER_PATH) if f.lower().endswith('.pdf')]

    # Filter out files that are already in the set
    files_to_grade = [f for f in files if f not in processed_files]

    print(f"Found {len(files)} total files.")
    print(f"Skipping {len(files) - len(files_to_grade)} already graded.")
    print(f"Starting grading for {len(files_to_grade)} remaining exams...")

    for i, filename in enumerate(files_to_grade):
        print(f"[{i+1}/{len(files_to_grade)}] Processing {filename}...")
        file_path = os.path.join(EXAM_FOLDER_PATH, filename)

        # 1. GRADE THE EXAM
        # (We use the retry logic from before to be safe)
        data = None
        attempts = 0
        while attempts < 3 and not data:
            try:
                data = grade_exam(file_path)
            except Exception as e:
                print(f"  Attempt {attempts+1} failed. Retrying...")
                time.sleep(5)
                attempts += 1

        # 2. APPEND TO CSV IMMEDIATELY
        if data:
            data['filename'] = filename

            # Convert single dictionary to a 1-row DataFrame
            single_row_df = pd.DataFrame([data])

            # Clean up columns (optional, to keep it tidy)
            cols = ['student_name', 'total_score', 'section_1_score', 'section_2_score', 'section_3_score', 'comments', 'scoring_breakdown', 'filename']
            # Ensure we only use columns that actually exist in the data
            present_cols = [c for c in cols if c in single_row_df.columns]
            single_row_df = single_row_df[present_cols]

            # APPEND TO FILE
            # mode='a' means append
            # header=... checks if file exists. If NO, write header. If YES, skip header.
            single_row_df.to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)

            print(f"  Saved {data['student_name']} to CSV.")
        else:
            print(f"  Skipping {filename} due to errors.")

        # Sleep to avoid rate limits
        time.sleep(4)

print("\nGrading Complete!")

NameError: name 'os' is not defined